In [ ]:
import numpy as np
import numpy.typing as npt
import networkx as nx
import matplotlib.pyplot as plt
import casadi as ca

In [ ]:
def calculate_pipe_flow_reynolds_number(mean_volume_flow: float, pipe_diameter: float, density: float, dynamic_viscosity: float) -> float:
    return (density*(mean_volume_flow/(np.pi*(pipe_diameter/2)**2))*pipe_diameter)/(dynamic_viscosity)

In [ ]:
def headloss_pipes(inductance: float | npt.NDArray, resistance: float | npt.NDArray, volume_flow: float | npt.NDArray) -> float | npt.NDArray:
    if not (inductance.shape == resistance.shape == volume_flow.shape):
        raise ValueError("Input values not of the same shape.")
    with np.errstate(divide='raise', invalid='raise'):
        return (resistance/inductance)*abs(volume_flow)*volume_flow

In [ ]:
def integrate(
    t, f, x_0, dt, args={}, tol=1e-15
):  # Runge-Kutta 45 https://de.wikipedia.org/wiki/Klassisches_Runge-Kutta-Verfahren
    k1 = f(t, x_0, **args)
    k2 = f(t + dt / 2, x_0 + dt / 2 * k1, **args)
    k3 = f(t + dt / 2, x_0 + dt / 2 * k2, **args)
    k4 = f(t + dt, x_0 + dt * k3, **args)
    return x_0 + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)

In [ ]:
def solve_continuous(t, x, A, u, B, F):
    x_dot = A @ x + B @ u - F
    return x_dot

In [ ]:
def fx(x, dt, t, A, B, u, F):
    x_new = integrate(t, solve_continuous, x, dt, args={"A": A, "B": B, "u": u, "F": F})
    return x_new

In [ ]:
## constants
# gravitational constant
g = 9.81  # m/s^2
# wave speed
a = 1.5e3  # m/s
# density
rho =  998.2  # kg/m^3
# dynamic viscosity
mu = 1.002e-3  # Ns/m^2

In [ ]:
## average operating conditions
# volume flow
q_mean = 0.0001#0224331792911098  # m^3/2

In [ ]:
internal_nodes = {
    "consumer_valve_inlet": {
        "name": "consumer_valve_inlet",
        "elevation": 0.875,
        "demand": 0,
    },
    "consumer_valve_outlet": {
        "name": "consumer_valve_outlet",
        "elevation": 0.875,
        "demand": 0,
    },
    "consumer_outlet": {
        "name": "consumer_outlet",
        "elevation": 0.83,
        "demand": 0.0001,  # 0224331792911098
    },
}

In [ ]:
reservoirs = {
        "tank_outlet": {
        "name": "tank_outlet",
        "elevation": 1.2,
        "pressure": 0.2,
    },
}

In [ ]:
pipes = {
    "tank_outlet_pipe": {
        "start_node": "tank_outlet",
        "end_node": "consumer_valve_inlet",
        "type": "pipe",
        "length": 1.7,
        "diameter": 0.015,
        "roughness": 0.007e-3,
        "friction_factor": 3.383e-3  # 2.65e-1,
    },
    "valve_pipe": {
        "start_node": "consumer_valve_inlet",
        "end_node": "consumer_valve_outlet",
        "type": "pipe",
        "length": 0.1,
        "diameter": 0.015,
        "roughness": 0.007e-3,
        "friction_factor": 1.65e-3  # 1.325e-1,  # 0.021
    },
    "consumer_outlet_pipe": {
        "start_node": "consumer_valve_outlet",
        "end_node": "consumer_outlet",
        "type": "pipe",
        "length": 0.35,
        "diameter": 0.015,
        "roughness": 0.007e-3,
        "friction_factor": 4.4e-3  # 1.1357142e-1,
    },
}

In [ ]:
pipe_number = len(pipes.keys())
internal_node_number = len(internal_nodes.keys())
reservoir_number = len(reservoirs.keys())

In [ ]:
for name, properties in pipes.items():
    properties["cross_section_area"] = (properties["diameter"]/2)**2 * np.pi

In [ ]:
inverse_relative_roughnesses = {}
for name, properties in pipes.items():
    inverse_relative_roughnesses[name] = properties["diameter"]/properties["roughness"]
inverse_relative_roughnesses

In [ ]:
reynolds_numbers = {}
for name, properties in pipes.items():
    reynolds_numbers[name] = calculate_pipe_flow_reynolds_number(q_mean, properties["diameter"], rho, mu)
reynolds_numbers

In [ ]:
initial_condition_flow_pipes = np.full((len(pipes.keys()),1),q_mean)

In [ ]:
graph = nx.Graph()
graph.add_nodes_from(reservoirs.keys())
graph.add_nodes_from(internal_nodes.keys())

In [ ]:
edge_dict = {}
for name, edge in pipes.items():
    edge_dict[name] = (edge["start_node"], edge["end_node"])
graph_edge_list = [edge for edge in edge_dict.values()]
graph_edge_list

In [ ]:
graph.add_edges_from(edge_dict.values())

In [ ]:
graph.edges

In [ ]:
nx.draw(graph)

In [ ]:
internal_nodes_list = [name for name in internal_nodes.keys()]
reservoir_nodes_list = [name for name in reservoirs.keys()]
node_list = internal_nodes_list + reservoir_nodes_list
node_list

In [ ]:
pipe_list = [name for name in pipes.keys()]
edge_list = pipe_list
edge_list

In [ ]:
incidence_matrix = -nx.incidence_matrix(graph, oriented=True, nodelist=node_list, edgelist=graph_edge_list)
print(incidence_matrix.todense())

In [ ]:
A_I = np.array(incidence_matrix.todense()[:internal_node_number])
A_R = np.array(incidence_matrix.todense()[-reservoir_number:])

In [ ]:
inductance = []
resistance = []
capacitance = []
for properties in pipes.values():
    inductance.append(properties["length"]/(g*properties["cross_section_area"]))
    resistance.append(8*properties["length"]*properties["roughness"]/(np.pi**2 * g * properties["diameter"]**5))
    capacitance.append((2*g * np.pi / 4 * properties["diameter"]**2 * properties["length"])/(a**2))

In [ ]:
L = np.array(inductance).reshape(len(list(pipes.keys())),1)
R = np.array(resistance).reshape(len(list(pipes.keys())),1)
C = np.array(capacitance).reshape(len(list(pipes.keys())),1)

In [ ]:
L

In [ ]:
R

In [ ]:
np.diagflat(C)

In [ ]:
f_p_initial = headloss_pipes(L, R, initial_condition_flow_pipes)
f_p_initial

In [ ]:
reservoir_total_head = reservoirs["tank_outlet"]["pressure"] + (reservoirs["tank_outlet"]["elevation"])
reservoir_total_head

In [ ]:
internal_nodes["consumer_outlet"]["initial_head"] = 1.3652680432885076  # internal_nodes["consumer_outlet"]["elevation"]  # (1/(2*g))*(q_mean/pipes["consumer_outlet_pipe"]["cross_section_area"])**2
internal_nodes["consumer_valve_outlet"]["initial_head"] = 1.3709220837892178  # (internal_nodes["consumer_valve_outlet"]["elevation"])  # internal_nodes["consumer_outlet"]["initial_head"] + R[2,0] * q_mean**2 +
internal_nodes["consumer_valve_inlet"]["initial_head"] = 1.372537523110129  # (internal_nodes["consumer_valve_inlet"]["elevation"])  # internal_nodes["consumer_valve_outlet"]["initial_head"] + R[1,0] * q_mean**2 +

In [ ]:
for name, properties in pipes.items():
    properties["friction_factor"] = (internal_nodes[properties["start_node"]]["initial_head"] - internal_nodes[properties["end_node"]]["initial_head"]) / ((properties["length"]/properties["diameter"])*(1/(2*g))*(q_mean/properties["cross_section_area"])**2) if properties["start_node"] in internal_nodes.keys() else (reservoirs["tank_outlet"]["pressure"] + (reservoirs["tank_outlet"]["elevation"]) - internal_nodes[properties["end_node"]]["initial_head"]) / ((properties["length"]/properties["diameter"])*(1/(2*g))*(q_mean/properties["cross_section_area"])**2)
    print(f"{name}: {properties["friction_factor"]}")

In [ ]:
h_I_0 = np.array([node["initial_head"] for node in internal_nodes.values()]).reshape(internal_node_number,1)
h_I_0

In [ ]:
h_R_0 = np.array([reservoir_total_head]).reshape(reservoir_number,1)
h_R_0

In [ ]:
Q_0 = np.array([node["demand"] for node in internal_nodes.values()]).reshape(internal_node_number,1)
Q_0

In [ ]:
G = np.linalg.inv(np.diagflat((0.5 * (abs(A_I) @ C))))
-G[2,2]

In [ ]:
# A = np.block([[np.zeros(shape=(pipe_number, pipe_number)), np.linalg.inv(np.diagflat(L)) @ A_I.transpose()],
#               [G @ A_I, np.zeros(shape=(internal_node_number, internal_node_number))],
#               ])
# -np.linalg.inv(np.diagflat(L)) @ np.diagflat(R) @ np.diagflat(np.full((1,pipe_number), np.abs(q_mean))[0])
A = np.block([[np.zeros(shape=(pipe_number, pipe_number)), np.linalg.inv(np.diagflat(L)) @ A_I.transpose()],[-G @ A_I, np.zeros(shape=(internal_node_number, internal_node_number))]])
A

In [ ]:
B = np.block([[np.linalg.inv(np.diagflat(L)) @ A_R.transpose(), np.zeros(shape=(pipe_number, internal_node_number))],
              [np.zeros(shape=(internal_node_number, reservoir_number)), -G],
              ])
B

In [ ]:
F = np.block([[(np.linalg.inv(np.diagflat(L)) @ np.diagflat(R) @ np.full((1,pipe_number), np.abs(q_mean)*q_mean)[0]).reshape(pipe_number,1)],
              [np.zeros(shape=(internal_node_number, 1))],
              ])
F

In [ ]:
x_0 = np.block([[np.full(shape=(pipe_number, 1), fill_value=q_mean)],
                [h_I_0],
                ])
x_0

In [ ]:
u_0 = np.concatenate((h_R_0, Q_0))
u_0

In [ ]:
B_u = B @ u_0
u_0
B_u[-1,0]

In [ ]:
dt = 1e-4
t=np.arange(0,25,dt)

In [ ]:
(A @ x_0)[0,]

In [ ]:
(B @ u_0)[0,]

In [ ]:
u = [u_0]*len(t)
x_hist = [x_0]
F_res = [F]
F_res[-1]

In [ ]:
for i in range(1,len(t)):
    x_hist.append(fx(x_hist[-1], dt, t[i], A, B, u[i], F_res[-1]))
    F_res.append(np.block([[np.linalg.inv(np.diagflat(L)) @ np.diagflat(R) @ x_hist[-1][:pipe_number]],
              [np.zeros(shape=(internal_node_number, 1))],
              ])
        )

In [ ]:
x_res = np.array(x_hist)
fig, axs = plt.subplots(len(x_0),figsize=(5,20))

for i in range(len(x_0)):
    axs[i].plot(t,x_res[:,i])


In [ ]:
x_hist[-1]

Casadi

In [ ]:
# Dynamic state
flow_tank_outlet_pipe = ca.MX.sym("flow_tank_outlet_pipe")
flow_valve_pipe = ca.MX.sym("flow_valve_pipe")
flow_consumer_outlet_pipe = ca.MX.sym("flow_consumer_outlet_pipe")
head_control_valve_inlet = ca.MX.sym("head_control_valve_inlet")
head_control_valve_outlet = ca.MX.sym("head_control_valve_outlet")
head_consumer_outlet = ca.MX.sym("head_consumer_outlet")
x = ca.vertcat(flow_tank_outlet_pipe, flow_consumer_outlet_pipe, head_control_valve_inlet, head_control_valve_outlet, head_consumer_outlet, valve_opening)
# x_dot_sym = ca.MX.sym("x_dot_sym", x.shape[0])
# x_dot = ca.vertcat(x_dot_sym)

In [ ]:
reservoir_head = ca.MX.sym("reservoir_head")
demand_control_valve_inlet = ca.MX.sym("demand_control_valve_inlet")
demand_control_valve_outlet = ca.MX.sym("demand_control_valve_outlet")
demand_consumer_outlet = ca.MX.sym("demand_consumer_outlet")
u = ca.vertcat(reservoir_head, demand_control_valve_inlet, demand_control_valve_outlet, demand_consumer_outlet)

In [ ]:
# Non-linear termns
F_p_tank_outlet_pipe = (np.linalg.inv(np.diagflat(L)) @ np.diagflat(R))[0,0] * ca.fabs(flow_tank_outlet_pipe)*flow_tank_outlet_pipe
F_p_flow_valve_pipe = (np.linalg.inv(np.diagflat(L)) @ np.diagflat(R))[0,0] * ca.fabs(flow_valve_pipe)*flow_valve_pipe
F_p_consumer_outlet_pipe = (np.linalg.inv(np.diagflat(L)) @ np.diagflat(R))[2,2] * ca.fabs(flow_consumer_outlet_pipe)*flow_consumer_outlet_pipe
F_p = ca.vertcat(F_p_tank_outlet_pipe, F_p_flow_valve_pipe, F_p_consumer_outlet_pipe, *([0]*internal_node_number))

In [ ]:
x_dot = A @ x + B @ u - F


In [ ]:
dt = 1e-4
t=np.arange(0,25,dt)
t0 = 0
len(t)

In [ ]:
dae = {
    "x":   x,
    "p":   u,
    "ode": x_dot,
}

In [ ]:
Fint = ca.integrator("Fint", "cvodes", dae, t0, dt, ) # {"calc_ic": True}

In [ ]:
# initial conditions
x0 = ca.DM(x_0)
u0 = ca.DM(u_0)

In [ ]:
x_vals = np.zeros((len(t), x0.numel()))
x_vals[0, :] = np.array(x0.T).ravel()
xt=x0
len(x_vals)

In [ ]:
for i in range(len(t)-1):
    res = Fint(x0=xt, p=u0)
    xt = res["xf"]
    x_vals[i+1, :] = np.array(xt.T).ravel()

In [ ]:
fig, axs = plt.subplots(len(x_0),figsize=(5,20))

for i in range(len(x_0)):
    axs[i].plot(t,x_vals[:,i])


In [ ]:
x_vals[-1,:]